# Part One
## Here we will do these things - 
 - #### Process Text
 - #### Clean Text
 - #### Tokenize the text and create sequences with keras 

RNN forgets the information trained which was trained earlier - after a while of training new and new data. 

Long short term memory (LSTM) cellw as created to help address these RNN issues. 


What does RNN do? 

It gives input - gets output - and gives this output as input to same network - this is time series data - 
so it checkes input for time instances 

In [1]:
def read_file(filepath):
    with open(filepath) as f:
        str_text = f.read()

    return str_text

In [21]:
# read_file("L:\\Workspace\\Resources\\UPDATED_NLP_COURSE\\06-Deep-Learning\\moby_dick_four_chapters.txt")

In [3]:
import spacy

In [6]:
nlp = spacy.load('en_core_web_lg', disable=['parser','tagger','ner'])

In [7]:
nlp.max_length = 1198623

In [8]:
# I wanna remove punctuations, because they occur often and we dont want out model to get into training
def sep_punc(doc_text):
    return [token.text.lower() for token in nlp(doc_text) if token.text not in '\n\n \n\n\n!"-#$%&()--.*+,-/:;<=>?@[\\]^_`{|}~\t\n ']

In [9]:
d = read_file("L:/Workspace/Resources/UPDATED_NLP_COURSE/06-Deep-Learning/moby_dick_four_chapters.txt")

In [10]:
tokens = sep_punc(d)

C:\Users\Asus\anaconda3\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [12]:
len(tokens)

11338

In [23]:
tokens[:30]

['call',
 'me',
 'ishmael',
 'some',
 'years',
 'ago',
 'never',
 'mind',
 'how',
 'long',
 'precisely',
 'having',
 'little',
 'or',
 'no',
 'money',
 'in',
 'my',
 'purse',
 'and',
 'nothing',
 'particular',
 'to',
 'interest',
 'me',
 'on',
 'shore',
 'i',
 'thought',
 'i']

In [13]:
# We will pass first 25 words of sentences, and we will let our model predict 26th. 

In [14]:
train_len = 25 +1 
text_seq = []
for i in range(train_len, len(tokens)):
    seq = tokens[i-train_len:i]
    text_seq.append(seq)

In [15]:
' '.join(text_seq[0])

'call me ishmael some years ago never mind how long precisely having little or no money in my purse and nothing particular to interest me on'

In [16]:
' '.join(text_seq[2])

'ishmael some years ago never mind how long precisely having little or no money in my purse and nothing particular to interest me on shore i'

# What did we do here? 
We made nlp object. Then we removed parser, tagger, and ner (named entity recognizer) from it. 
Then we made a tokens through the text file - how? By removing the puncutations and new lines - because we dont wanna make them our training. 
Then now - we took 25 words, and we ask what is next word? Its nto doing any work, just checking in the list of tokens, and giving us another word. That's all. 

In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [24]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(text_seq)

In [25]:
sequences = tokenizer.texts_to_sequences(text_seq)

In [32]:
# tokenizer.index_word

We simply gave tokenizer the text sequences we created - and then we got tokens - id for each word 

In [45]:
vocab_size = len(tokenizer.word_counts) # Tells us how many times a word comes 

In [29]:
len(tokenizer.word_counts)

2718

In [33]:
len(sequences)

11312

In [31]:
import numpy as np
sequences = np.array(sequences)
sequences

array([[ 956,   14,  263, ..., 2713,   14,   24],
       [  14,  263,   51, ...,   14,   24,  957],
       [ 263,   51,  261, ...,   24,  957,    5],
       ...,
       [ 952,   12,  166, ...,  262,   53,    2],
       [  12,  166, 2712, ...,   53,    2, 2718],
       [ 166, 2712,    3, ...,    2, 2718,   26]])

# Part Two - 
- #### In this part we will create LSTM.
- #### Split the data.
- #### And fit our model to the data 

In [34]:
from keras.utils import to_categorical

Now how to get our X and y? 
So for X we need every item of our OUTER array - and every time except last one on our inner array -  

That means - 
sequences[: , :-1] - meaning take all the sequences - and in all those take everything except last item 

In [49]:
X = sequences[:, :-1]

Now we take y - we take all the sequences from outer array . 
- sequences[:, :] - 
but in this we take only last character - or last item for our y 
- sequences(:,-1]

In [40]:
y = sequences[:,-1]

In [41]:
y

array([  24,  957,    5, ...,    2, 2718,   26])

In [46]:
y = to_categorical(y, num_classes=vocab_size+1)

In [50]:
seq_len = X.shape[1]

In [52]:
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding

In [ ]:
def create_model(vocab_size, seq_len):
    model = Sequential()
    model.add(Embedding(vocab_size, seq_len, input=seq_len))
    # we are giving input_dimension - vocab_size , and output_dimension = seq_len, and input_len 
    model.add(LSTM((Seq_len*3), return_sequences=True))
    # We added 3 times out sequence len 
    model.add(LSTM(50))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(vocab_size, activation='softmax'))

    model.compile(